In [ ]:
from lightglue import LightGlue, SuperPoint, DISK
from lightglue.utils import load_image, rbd
from lightglue import viz2d
import torch

torch.set_grad_enabled(False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # 'mps', 'cpu'

extractor = SuperPoint(max_num_keypoints=2048).eval().to(device)  # load the extractor
matcher = LightGlue(features="superpoint").eval().to(device)

# extractor = DISK(max_num_keypoints=2048).eval().cuda()  # load the extractor
# matcher = LightGlue(features='disk').eval().cuda()  # load the matcher

In [ ]:
from diffdrr.drr import DRR
from diffdrr.data import load_example_ct
from diffdrr.visualization import plot_drr
import torch
import math

subject = load_example_ct(bone_attenuation_multiplier=6)

# subject.density = subject.volume
print(subject.density.shape)

# We are going to create 2 views with 45° rotation between both

height = 256
sdd = 1020
delx = 2.0
trans = 850.0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

drr_trilinear = DRR(
    subject,     # An object storing the CT volume, origin, and voxel spacing
    sdd=sdd,  # Source-to-detector distance (i.e., focal length)
    height=height,  # Image height (if width is not provided, the generated DRR is square)
    delx=delx,    # Pixel spacing (in mm)
    renderer='trilinear',
    compile=False
).to(device)

rotations = torch.tensor([[0, 0, 0], [0, 0, 0], [0, 0, 0]], device=device)
translations = torch.tensor([[0.0, trans, 0.0], [0.0, 0.8*trans, 0.0], [0.0, 1.1*trans, 0.0]], device=device)

with torch.no_grad():
    img = drr_trilinear(rotations, translations, parameterization="euler_angles", convention="ZXY")

plot_drr(img, ticks=False)

In [ ]:
from pose_estimate_from_keypoints_opencv import (
    estimate_pose,
    estimate_reprojection_error,
    estimate_3d_points
)
import numpy as np

print(img.shape)
print(img.min(), img.max()) # coming out of DiffDRR: [0, 22]

img_normalized = img / img.max()

print(img_normalized.min(), img_normalized.max()) # coming out of DiffDRR: [0, 22]

image0 = img_normalized[0]
image1 = img_normalized[1]


feats0 = extractor.extract(image0.to(device))
feats1 = extractor.extract(image1.to(device))

matches01 = matcher({"image0": feats0, "image1": feats1})
feats0, feats1, matches01 = [
    rbd(x) for x in [feats0, feats1, matches01]
]  # remove batch dimension

kpts0, kpts1, matches = feats0["keypoints"], feats1["keypoints"], matches01["matches"]
m_kpts0, m_kpts1 = kpts0[matches[..., 0]], kpts1[matches[..., 1]]

axes = viz2d.plot_images([image0, image1])
viz2d.plot_matches(m_kpts0, m_kpts1, color="lime", lw=0.2)
viz2d.add_text(0, f'Stop after {matches01["stop"]} layers', fs=20)

kpc0, kpc1 = viz2d.cm_prune(matches01["prune0"]), viz2d.cm_prune(matches01["prune1"])
viz2d.plot_images([image0, image1])
viz2d.plot_keypoints([kpts0, kpts1], colors=[kpc0, kpc1], ps=10)

# From Tutorial with Gemini
# _, h, w = image0.shape
# focal_length = w  # A reasonable guess
# center = (w / 2, h / 2)
# camera_matrix = np.array(
#     [[focal_length, 0, center[0]],
#      [0, focal_length, center[1]],
#      [0, 0, 1]], dtype=np.float32
# )
# mkpts0 = m_kpts0.cpu().numpy()
# mkpts1 = m_kpts1.cpu().numpy()
# R, t = estimate_pose(camera_matrix, mkpts0, mkpts1)
# reprojection_error = estimate_reprojection_error(mkpts0, mkpts1, camera_matrix, R, t)
# print(f"\nMean Reprojection Error: {reprojection_error:.4f} pixels")

# drr_pts3d = estimate_3d_points(mkpts0, mkpts1, camera_matrix, R, t)

In [ ]:
# Estimate 3D Points
from diffdrr.pose import RigidTransform, convert
import cv2

# These are the poses that transform from camera frame to world (CT) frame
poses = convert(rotations, translations, parameterization="euler_angles", convention="ZXY")
print(poses.matrix.shape)
pose1 = poses.matrix[0]
pose2 = poses.matrix[1]

# Intrinsic Matrix K (3x3)
K_torch = drr_trilinear.detector.intrinsic
K_np = K_torch.cpu().numpy()


# # Extrinsic Matrices [R|t] (world-to-camera)
# # This is the INVERSE of the generation pose
extrinsics = drr_trilinear.detector.reorient.compose(poses).inverse()
extrinsic1_torch = extrinsics.matrix[0]
extrinsic2_torch = extrinsics.matrix[1]
# The extrinsic matrix for OpenCV is the first 3 rows of this 4x4 matrix
extrinsic1_np = extrinsic1_torch[:3, :].cpu().numpy()
extrinsic2_np = extrinsic2_torch[:3, :].cpu().numpy()


# --- 3. Construct the final Projection Matrices (P1, P2) ---
# P = K @ [R|t]
P1 = K_np @ extrinsic1_np
P2 = K_np @ extrinsic2_np

print("Successfully extracted Projection Matrix P1:\n", P1)
print("\nSuccessfully extracted Projection Matrix P2:\n", P2)


# --- 4. Find 2D Correspondences ---
# As before, use LightGlue to get `points1` and `points2` between the
# two DRRs generated with `pose1` and `pose2`.
# (Using dummy points for the example)
width, height = drr_trilinear.detector.width, drr_trilinear.detector.height

points1 = m_kpts0.cpu().numpy()
points2 = m_kpts1.cpu().numpy()
points1_for_cv = points1.T.astype(np.float32)
points2_for_cv = points2.T.astype(np.float32)

points_4d_hom = cv2.triangulatePoints(P1, P2, points1_for_cv, points2_for_cv)
points_3d = points_4d_hom / points_4d_hom[3]
points_3d = points_3d[:3, :].T

In [ ]:
# # Torch Triangulation
# height, width = drr_trilinear.detector.height, drr_trilinear.detector.width
# poses = convert(rotations, translations, parameterization="euler_angles", convention="ZXY")
# sources, targets = drr_trilinear.detector(poses, calibration=None)
# print(sources.shape, targets.shape)
# idx1 = points1_px_int[:, 1] * width + points1_px_int[:, 0]
# idx2 = points2_px_int[:, 1] * width + points2_px_int[:, 0]
# Compute in Torch directly the sources, and targets

In [ ]:
# visualize the 3d points with the volume!
import matplotlib.pyplot as plt
import pygmich.hcimg_core.computer_vision as cv
import k3d

print(points_3d.min(axis=0), points_3d.max(axis=0))
print(subject.density.data.min(), subject.density.data.max())

# vol = subject.density.data.cpu().numpy().squeeze()

bones = (subject.volume.data > 350).cpu().numpy().squeeze()
affine_matrix = subject.volume.affine

vertices_voxel, faces = cv.mask_to_mesh(bones)
vertices_voxel_xyz = vertices_voxel[:,::-1].copy()
num_vertices = vertices_voxel_xyz.shape[0]
vertices_voxel_hom = np.hstack((vertices_voxel_xyz, np.ones((num_vertices, 1))))
#transformed_vertices_hom = (affine_matrix @ vertices_voxel_hom.T).T
transformed_vertices_hom = (vertices_voxel_hom @ affine_matrix.T)
# transformed_vertices_hom = (affine_matrix @ vertices_voxel_hom.T).T
vertices_world = transformed_vertices_hom[:, :3]

print(vertices_world.min(0), vertices_world.max(0))
mesh = k3d.mesh(vertices_world.astype(np.float32).tolist(), faces.astype(np.uint32).tolist(), color=0xff0000, opacity=0.8, name="bones")

# # print(vol.shape)

# # vol_zyx = np.moveaxis(vol, 2, 0)[::-1].copy()

# # print(vol_zyx.shape)
# # plt.imshow(vol_zyx.max(axis=2)>0.2)
# # plt.show()

# # print(vol.shape)
plot = k3d.plot()
plot += k3d.points(points_3d, point_size=6, color=0x00ff00)
plot += mesh
plot.display()

In [ ]:
# Now Create correspondances between image0,1 with 2 (the surrogate Fluroscopy)
print(img.shape)
print(img_normalized.shape)

pseudo_fluoro = img_normalized[2]


feats0 = extractor.extract(image0.to(device))
feats3 = extractor.extract(pseudo_fluoro.to(device))
matches03 = matcher({"image0": feats0, "image1": feats3})


feats0, feats3, matches03 = [
    rbd(x) for x in [feats0, feats3, matches03]
]  # remove batch dimension

kpts0, kpts3, matches = feats0["keypoints"], feats3["keypoints"], matches03["matches"]
m_kpts0, m_kpts3 = kpts0[matches[..., 0]], kpts3[matches[..., 1]]

print(m_kpts0.shape)
axes = viz2d.plot_images([image0, pseudo_fluoro])
viz2d.plot_matches(m_kpts0.reshape(-1,2), m_kpts3.reshape(-1,2), color="lime", lw=0.2)
# viz2d.add_text(0, f'Stop after {matches01["stop"]} layers', fs=20)

